<!-- NOTEBOOK_METADATA source: "⚠️ Jupyter Notebook" title: "Bifrost AI Gateway Integration" sidebarTitle: "Bifrost" logo: "/images/integrations/bifrost_icon.png" description: "Learn how to send OpenTelemetry traces from the Bifrost AI gateway to Langfuse while using its OpenAI-compatible API." category: "Integrations" -->

# Integrate Langfuse with Bifrost

This notebook shows how to connect Langfuse to the Bifrost AI gateway for tracing and observability of OpenAI-compatible LLM requests.

> **What is Bifrost?** [Bifrost](https://github.com/maximhq/bifrost) is an open-source, self-hosted Go AI gateway that provides an OpenAI-compatible API with multi-provider routing, adaptive load balancing, guardrails, and virtual keys.

> **What is Langfuse?** [Langfuse](https://langfuse.com) is an open-source LLM engineering platform that helps teams trace, debug, and evaluate their LLM applications.

<!-- STEPS_START -->
## Step 1: Install Dependencies

In [ ]:
%pip install langfuse openai -U

## Step 2: Set Up Environment Variables

Get your Langfuse keys from the project settings in [Langfuse Cloud](https://langfuse.com/cloud) or set up [self-hosting](https://langfuse.com/self-hosting).

In [ ]:
import os

# Get keys for your project from the project settings page: https://langfuse.com/cloud
os.environ.setdefault("LANGFUSE_PUBLIC_KEY", "pk-lf-...");
os.environ.setdefault("LANGFUSE_SECRET_KEY", "sk-lf-...");
os.environ.setdefault("LANGFUSE_BASE_URL", "https://cloud.langfuse.com"); # 🇪🇺 EU region (API host)
# Other Langfuse data regions include 🇺🇸 US: https://us.cloud.langfuse.com, 🇯🇵 Japan: https://jp.cloud.langfuse.com and ⚕️ HIPAA: https://hipaa.cloud.langfuse.com

os.environ.setdefault("BIFROST_BASE_URL", "http://localhost:8080/v1");  # Bifrost's OpenAI-compatible base URL
os.environ.setdefault("BIFROST_API_KEY", "your-bifrost-virtual-key");  # A Bifrost virtual key with access to the selected model

With the environment variables set, initialize the Langfuse client. `get_client()` picks up the env vars above and returns a client bound to your project.


In [ ]:
from langfuse import get_client

langfuse = get_client()

# Verify connection
if langfuse.auth_check():
    print("Langfuse client is authenticated and ready!")
else:
    print("Authentication failed. Please check your credentials and host.")

## Step 3: Configure Bifrost OpenTelemetry Export

In [ ]:
import base64
import json
import os
from pathlib import Path

# Langfuse's OTLP endpoint uses Basic Auth with the project public and secret keys.
langfuse_auth = base64.b64encode(
    f"{os.environ['LANGFUSE_PUBLIC_KEY']}:{os.environ['LANGFUSE_SECRET_KEY']}".encode()
).decode()
os.environ['LANGFUSE_OTEL_AUTH'] = f"Basic {langfuse_auth}"

bifrost_config = {
    "plugins": [
        {
            "enabled": True,
            "name": "otel",
            "config": {
                "service_name": "bifrost",
                "collector_url": f"{os.environ['LANGFUSE_BASE_URL']}/api/public/otel/v1/traces",
                "trace_type": "genai_extension",
                "protocol": "http",
                "headers": {
                    "Authorization": "env.LANGFUSE_OTEL_AUTH",
                    "x-langfuse-ingestion-version": "4"
                }
            }
        }
    ]
}

Path("bifrost-config.json").write_text(
    json.dumps(bifrost_config, indent=2), encoding="utf-8"
)
print("Wrote bifrost-config.json. Start Bifrost with this configuration before running the next step.")

## Step 4: Send a Request Through Bifrost

In [ ]:
import os

from openai import OpenAI

# Bifrost routes this OpenAI-compatible request to the configured provider.
client = OpenAI(
    base_url=os.environ["BIFROST_BASE_URL"],
    api_key=os.environ["BIFROST_API_KEY"],
)

response = client.chat.completions.create(
    model="openai/gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Explain why tracing routed LLM requests is useful."}
    ],
)

print(response.choices[0].message.content)

## Step 5: View Traces in Langfuse

After running the example, open [Langfuse Cloud](https://langfuse.com/cloud) to see the full trace including prompts, completions, tool calls, token usage, and latency.

![Example Bifrost trace in Langfuse](https://langfuse.com/images/cookbook/integration-bifrost/bifrost-example-trace.png)

[Example trace in Langfuse](TODO: replace with a public Bifrost trace URL from your Langfuse project)

<!-- STEPS_END -->

<!-- MARKDOWN_COMPONENT name: "LearnMore" path: "@/components-mdx/integration-learn-more.mdx" -->